# Visualize Learned CNN Filters

Convolution filters are trainable weights. After training, the first layer often learns simple stroke and edge detectors.

## Initialize PyTorch

- Import PyTorch, torchvision, and plotting tools
- Seed PyTorch for reproducibility
- Detect device (GPU, Apple Silicon, CPU)

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)

def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

device = get_device()

## Define CNN Model

- Build a CNN with named convolution layers
- Keep `conv1` accessible for filter visualization
- Prepare a classifier for MNIST digits

In [ ]:
class MNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.relu = nn.ReLU()
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(16 * 7 * 7, 64), nn.ReLU(), nn.Linear(64, 10))

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        return self.classifier(x)

model = MNISTCNN().to(device)

## Plot Initial Filters

- Read the first convolution layer weights
- Display each filter before training
- Use this as a baseline for random initialization

In [ ]:
def plot_conv1_filters(model, title):
    weights = model.conv1.weight.detach().cpu().squeeze(1)
    fig, axes = plt.subplots(1, len(weights), figsize=(12, 2))
    fig.suptitle(title)
    for ax, filt in zip(axes, weights):
        ax.imshow(filt, cmap="coolwarm")
        ax.axis("off")
    plt.tight_layout()

plot_conv1_filters(model, "conv1 filters before training")

## Train CNN

- Load MNIST training data
- Train the model for a few epochs
- Let convolution filters adapt to digit patterns

In [ ]:
transform = transforms.ToTensor()
train_data = datasets.MNIST(root="../../data", train=True, download=True, transform=transform)
train_loader = DataLoader(Subset(train_data, range(5000)), batch_size=64, shuffle=True)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(2):
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        loss = loss_fn(model(images), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

## Plot Learned Filters

- Display first-layer filters after training
- Compare learned filters against the random starting filters
- Discuss how filters become useful pattern detectors

In [ ]:
plot_conv1_filters(model, "conv1 filters after training")

## Exercise

Change the number of output channels in `conv1`. How does the filter visualization change?